# Task 4 — YOLOv8 5-fold + masked pool (Kaggle판, fold 분담 학습용)

**목적**: Colab판(`task4_yolov8_5fold_masked_colab`)과 **완전히 동일한 데이터·설정**으로, Kaggle GPU에서
지정한 fold만 학습해 체크포인트를 분담 생산한다. **fold0는 Colab에서 진행 중**이므로 이 노트북은
기본적으로 fold1~4를 담당한다 (학습할 fold는 14번 셀 `FOLDS_THIS_SESSION`으로 지정).

| 항목 | 내용 |
|---|---|
| 데이터/분할/loss | Colab판과 동일: task2-masked 데이터(원본+pool2+masked, `syn_00505/00102` 제외, corrections 구버전 스냅샷) + `fold_split_masked.json` 고정 분할 + YOLOv8m, imgsz 960, cls gain **1.5** |
| 체크포인트 | `{tag}_fold{i}_best.pt` — **Colab판과 파일명 체계 동일** → 나중에 한 폴더에 모으면 앙상블 노트북에서 그대로 사용 |
| 세션 전략 | 세션당 fold 1개 권장 (12h 한도). 완료 fold는 `backup_dir`에 best.pt가 있으면 자동 스킵 |

**실행 전제 (Input 연결 4가지)**
- competition 데이터(`train_images`, `train_annotations`)
- 합성 pool(`task2_synthesized`) + masked pool(`dataset-74-masked`)
- **`fold_split_masked.json`** — Drive에 있는 파일을 그대로 **Kaggle Dataset으로 업로드**해 Input으로 연결
  (또는 이 json을 output에 포함한 task2-masked 커밋의 Output을 Input으로 연결. 파일명만 유지되면 자동 검색됨)

**운영 방법**
1. 14번 셀에서 `FOLDS_THIS_SESSION = [1]` 처럼 이번 세션에 학습할 fold를 지정
2. **Save Version(Save & Run All)** 으로 배치 실행 (인터랙티브 장시간 실행은 연결 끊김 위험)
3. 커밋 완료 후 Output의 `outputs/*_best.pt` 확인 → 다음 세션에서 다음 fold 지정 후 반복
4. 이미 완료한 fold의 체크포인트를 Input으로 연결하고 아래 선택 셀로 복사해두면 실수로 같은 fold를
   다시 돌려도 자동 스킵됨

**Kaggle Notebook: GPU on / Internet on** (ultralytics 설치 + yolov8m.pt 다운로드에 필요)


In [ ]:
# [1. 환경 준비] ultralytics(YOLOv8 학습 + COCO->YOLO 변환 유틸 포함)
#  ⚠ pip 설치는 세션이 재시작되면 사라지므로 매 세션 이 셀부터 실행해야 합니다.
!pip install -q "ultralytics>=8.3"

# 설치 검증: -q 옵션이 에러 로그를 가리는 경우 대비
import importlib
importlib.import_module('ultralytics')
print('설치 확인 OK: ultralytics')


In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다. (YOLO 관련 로직은 이 노트북에서 로컬로 정의)
import os, sys, re, glob, json, shutil, math
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/kaggle/working/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 (데이터 전처리 - 모델과 무관한 순수 로직) ──
from dataset import (
    load_raw_annotations, # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,    # corrections 기반 bbox 수정
    # make_folds는 일부러 import하지 않음 - fold 분할은 fold_split_masked.json 로드로 고정 (재계산 금지)
    cache_images,         # 이미지 로컬 캐시
    write_fold_dirs,      # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,       # label_map.json 저장
)

from ultralytics import YOLO
from ultralytics.data.converter import convert_coco   # COCO json -> YOLO txt 공식 변환 유틸

print('저장소 clone 완료:', REPO_DIR)


In [ ]:
# [3. corrections 하드코딩] task2-masked(Kaggle) 실험과 완전히 동일한 스냅샷을 사용합니다.
#  ⚠ 저장소 canonical corrections.json과 일부 다른 "구버전" 스냅샷입니다
#     (fix_category "3444"->3351 및 "791"->31863 포함, add_boxes 1건 category 상이).
#     fold-matched WBF 앙상블 파트너인 task2-masked RF-DETR 체크포인트와 학습 라벨 조건을
#     완전히 일치시키기 위해 의도적으로 이 스냅샷을 고정합니다.
#  apply_corrections()는 "파일 경로"를 받는 시그니처라(저장소 함수 무수정 원칙),
#  dict를 로컬 파일로 1회 저장한 뒤 그 경로를 넘기는 방식으로 우회합니다.
corrections = {
    "coord_fix": {
        "K-003351-016262-018357_0_2_0_2_75_000_200.png": [
            {"original": [6567, 625, 311, 315], "corrected": [567, 625, 311, 315]}
        ]
    },
    "remove_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [88, 255, 366, 209]}
        ]
    },
    "modify_boxes": {
        "K-003351-020014-020238_0_2_0_2_90_000_200.png": [
            {"category_id": 3351, "match_bbox": None, "new_bbox": [390, 260, 170, 165]}
        ],
        "K-003351-019232-029667_0_2_1_2_70_000_200.png": [
            {"category_id": 19232, "match_bbox": None, "directive": "EXTEND_DOWN_95"}
        ]
    },
    "add_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [90, 870, 245, 240]}
        ],
        "K-003351-013900-021325_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [400, 830, 180, 180]}
        ],
        "K-003351-013900-036637_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [440, 880, 175, 175]}
        ],
        "K-003351-020014-022074_0_2_0_2_90_000_200.png": [
            {"category_id": 20014, "bbox": [65, 720, 325, 315]}
        ],
        "K-003351-020238-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 20238, "bbox": [580, 290, 215, 215]}
        ],
        "K-003351-021325-032310_0_2_0_2_90_000_200.png": [
            {"category_id": 32310, "bbox": [595, 830, 345, 245]}
        ],
        "K-003351-029667-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [375, 870, 165, 165]}
        ],
        "K-003351-032310-038162_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [390, 855, 185, 185]}
        ],
        "K-003351-033880-038162_0_2_0_2_75_000_200.png": [
            {"category_id": 33880, "bbox": [70, 600, 310, 425]}
        ],
        "K-003351-035206-041768_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [460, 875, 180, 180]}
        ],
        "K-003544-004543-012247-016548_0_2_0_2_90_000_200.png": [
            {"category_id": 4543, "bbox": [640, 195, 205, 190]}
        ]
    },
    "fix_category": {
        "791": 31863,
        "3444": 3351,
        "3441": 3351,
        "1420": 35206,
        "1412": 27733
    }
}

CORRECTIONS_PATH = '/kaggle/working/corrections.json'
with open(CORRECTIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump(corrections, f, ensure_ascii=False, indent=2)
print('corrections 저장 (task2-masked 동일 스냅샷):', CORRECTIONS_PATH)


In [ ]:
# [4. 입력 경로 자동 탐색] (Kaggle)
#  /kaggle/input 아래에서 폴더 "이름"으로 찾기 때문에 dataset 슬러그를 몰라도 동작합니다.
def find_input_dir(name):
    """/kaggle/input 아래에서 이름이 name인 디렉토리를 찾아 반환합니다 (여러 개면 첫 번째)."""
    hits = sorted(p for p in glob.glob(os.path.join('/kaggle/input', '**', name), recursive=True)
                  if os.path.isdir(p))
    if len(hits) > 1:
        print(f"'{name}' 후보 {len(hits)}개 -> 첫 번째 사용:\n  " + "\n  ".join(hits))
    return hits[0] if hits else None

TRAIN_IMG_DIR = find_input_dir('train_images')       # 원본 train 이미지 (png)
TRAIN_ANN_DIR = find_input_dir('train_annotations')  # 원본 annotation (박스당 json 1개)

POOL_DIR      = find_input_dir('task2_synthesized')   # 합성 pool 루트 (images/, annotations/)
POOL_IMG_DIR  = os.path.join(POOL_DIR, 'images') if POOL_DIR else None
POOL_ANN_PATH = ((glob.glob(os.path.join(POOL_DIR, 'annotations', '*.json')) or [None])[0]
                 if POOL_DIR else None)

MASKED_DIR      = find_input_dir('dataset-74-masked')  # masked pool 루트 (images/, annotations/)
MASKED_IMG_DIR  = os.path.join(MASKED_DIR, 'images') if MASKED_DIR else None
MASKED_ANN_PATH = ((glob.glob(os.path.join(MASKED_DIR, 'annotations', '*.json')) or [None])[0]
                   if MASKED_DIR else None)

# 고정 fold 분할: 파일명으로 /kaggle/input 전체 재귀 검색 (Dataset이든 이전 커밋 Output이든 무관)
_hits = glob.glob('/kaggle/input/**/fold_split_masked.json', recursive=True)
assert _hits, ("fold_split_masked.json을 /kaggle/input 아래에서 못 찾음 - "
               "json을 담은 Dataset(또는 커밋 Output)이 Input으로 연결됐는지 확인.\n"
               f"현재 연결된 Input 폴더: {sorted(os.listdir('/kaggle/input'))}")
if len(_hits) > 1:
    print(f'fold_split_masked.json 후보 {len(_hits)}개 -> 첫 번째 사용:\n  ' + '\n  '.join(_hits))
FOLD_SPLIT_JSON = _hits[0]

paths = {'TRAIN_IMG_DIR': TRAIN_IMG_DIR, 'TRAIN_ANN_DIR': TRAIN_ANN_DIR,
         'POOL_IMG_DIR': POOL_IMG_DIR, 'POOL_ANN_PATH': POOL_ANN_PATH,
         'MASKED_IMG_DIR': MASKED_IMG_DIR, 'MASKED_ANN_PATH': MASKED_ANN_PATH,
         'FOLD_SPLIT_JSON': FOLD_SPLIT_JSON}
for k, v in paths.items():
    print(f'{k:16s}:', v)
assert all(paths.values()), "자동 탐색에 실패한 경로가 있습니다. 위 상수에 직접 경로를 지정하세요."


In [ ]:
# [5. 실험 설정] Colab판과 완전히 동일한 하이퍼파라미터 - 경로만 Kaggle 기준입니다.
#  tag도 동일하게 유지: fold별 체크포인트 파일명이 Colab 산출물과 같은 체계가 되어
#  나중에 한 폴더에 모아 앙상블 노트북에서 그대로 사용할 수 있습니다.
TASK_ID = 4
TAG = 'yolov8m_task4_syn74_masked'

config = {
    'data': {
        'n_splits': 5,
        'seed': 42,          # (참고) fold 분할은 fold_split_masked.json 로드로 고정 - seed는 분할에 관여하지 않음
        'dataset_dir': '/kaggle/tmp/dataset',             # COCO 포맷 fold 디렉토리
        'cache_dir': '/kaggle/tmp/img_cache',
        'yolo_dataset_dir': '/kaggle/tmp/yolo_dataset',   # YOLO 포맷(images/labels) fold 디렉토리
    },
    'model': {
        'variant': 'medium',   # yolov8m
        'tag': TAG,
    },
    'train': {
        'epochs': 50,
        'imgsz': 960,
        'batch': 8,             # Kaggle 16GB GPU 기준 (OOM 시 4로 축소)
        'patience': 10,         # val 개선이 10ep 없으면 조기 종료
        'seed': 42,
        'box_gain': 7.5,        # 기본값 유지
        'cls_gain': 1.5,        # 기본 0.5의 3배 - classification loss 패널티 강화 (Colab판과 동일)
        'dfl_gain': 1.5,        # 기본값 유지
    },
    'output': {
        'local_output_dir': '/kaggle/tmp/outputs',
        'backup_dir': '/kaggle/working/outputs',   # 체크포인트/학습곡선 (커밋 시 보존)
    },
}

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['data']['yolo_dataset_dir'], config['output']['local_output_dir'],
          config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 가속기 설정 확인')
print('backup_dir(커밋 시 보존):', config['output']['backup_dir'])
print('loss gains:', {k: v for k, v in config['train'].items() if k.endswith('_gain')})


In [ ]:
# [6. 원본 train 로드 + annotation 수정] Task2와 완전히 동일한 절차입니다.
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(TRAIN_ANN_DIR)
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))


In [ ]:
# [7. pool2 병합] Task2(Kaggle) 노트북과 완전히 동일한 로직입니다.
#  pool의 _annotations.coco.json은 "라벨 네임스페이스"(id 1~74, name=원본 category_id)로 작성되어 있어,
#  categories의 name 필드로 원본 id 공간으로 되돌린 뒤 병합합니다.
#  ⚠ fold 구성을 Task2와 일치시켜야 하므로, 이 병합 로직은 Task2 노트북과 토씨 하나 다르지 않게 유지합니다.
def load_pool_annotations(pool_ann_path):
    """합성 pool COCO json -> (boxes, cats(원본 id 공간), img_meta, 원본 coco dict)"""
    with open(pool_ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    label2cat_pool = {c['id']: int(c['name']) for c in coco['categories'] if c['id'] != 0}
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    p_boxes, p_cats, p_meta = defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        p_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        p_boxes[fn].append([float(v) for v in a['bbox']])
        p_cats[fn].append(label2cat_pool[a['category_id']])
    return p_boxes, p_cats, p_meta, coco

pool_boxes, pool_cats, pool_meta, pool_coco = load_pool_annotations(POOL_ANN_PATH)
print(f"pool2: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

overlap = set(pool_meta) & set(boxes_by_image)
assert not overlap, f'train/pool 파일명 충돌: {sorted(overlap)[:5]}'

empty_pool = [fn for fn in pool_meta if not pool_boxes.get(fn)]
if empty_pool:
    print(f'박스 0개인 pool 이미지 {len(empty_pool)}장 제외:', empty_pool[:5])

for fn in pool_meta:
    if pool_boxes.get(fn):
        boxes_by_image[fn] = pool_boxes[fn]
        cats_by_image[fn] = pool_cats[fn]
        img_meta[fn] = pool_meta[fn]

print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [7-1. masked pool 병합] dataset-74-masked (실사 기반 masked pool)
#  task2-masked(Kaggle) 노트북과 동일한 로직입니다. 요점:
#   - masked pool의 _annotations.coco.json은 categories id가 "원본 category_id 그대로"라 name 매핑 없이 사용
#   - 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있어, 전체 파일을 'msk_' 접두어로
#     리네임한 사본을 스테이징 폴더에 생성 (cache_images 덮어쓰기·corrections 오적용 방지)
#   - 같은 구성(위도/촬영조건 변형 + masked 버전)의 split 누수 방지 그룹화는 fold_split_masked.json에
#     이미 반영되어 있고, 9번 셀에서 그룹 누수 assert로 재검증합니다
MASKED_STAGE_DIR = '/kaggle/tmp/masked_renamed'
os.makedirs(MASKED_STAGE_DIR, exist_ok=True)

def load_masked_annotations(ann_path):
    """masked pool COCO json -> (boxes, cats, meta, coco). category_id를 원본 id로 그대로 사용합니다."""
    with open(ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    m_boxes, m_cats, m_meta = defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        m_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        m_boxes[fn].append([float(v) for v in a['bbox']])
        m_cats[fn].append(int(a['category_id']))
    return m_boxes, m_cats, m_meta, coco

masked_boxes, masked_cats, masked_meta, masked_coco = load_masked_annotations(MASKED_ANN_PATH)
print(f"masked pool: 이미지 {len(masked_meta)}장 / 박스 {sum(len(v) for v in masked_boxes.values())}개"
      f" / 클래스 {len({c for cs in masked_cats.values() for c in cs})}종")

# 안전장치 1: masked pool의 category_id가 74종 매핑(원본 id 공간 = pool2 JSON의 name)에 전부 포함되는지 확인
cats_74 = {int(c['name']) for c in pool_coco['categories'] if c['id'] != 0}
unknown = sorted({c for cs in masked_cats.values() for c in cs} - cats_74)
assert not unknown, f'74종 매핑에 없는 masked pool category_id: {unknown} (원본 id/라벨 공간 착오 확인)'

# 안전장치 2: annotation에는 있는데 실제 이미지 파일이 없는 항목 확인
masked_img_src = {os.path.basename(p): p
                  for p in glob.glob(os.path.join(MASKED_IMG_DIR, '**', '*.png'), recursive=True)}
missing_img = sorted(set(masked_meta) - set(masked_img_src))
assert not missing_img, f'annotation에는 있는데 이미지가 없는 파일: {missing_img[:5]}'

# 안전장치 3: 박스 0개 이미지는 제외 (task2-masked와 동일)
empty_masked = [fn for fn in masked_meta if not masked_boxes.get(fn)]
if empty_masked:
    print(f'박스 0개인 masked 이미지 {len(empty_masked)}장 제외:', empty_masked[:5])

# 리네임('msk_' + 원본 파일명) + 스테이징 복사(Drive -> 로컬) + annotation 병합
n_merged = 0
for fn in masked_meta:
    if not masked_boxes.get(fn):
        continue
    new_fn = 'msk_' + fn
    assert new_fn not in boxes_by_image, f'리네임 후에도 파일명 충돌: {new_fn}'
    dst = os.path.join(MASKED_STAGE_DIR, new_fn)
    if not os.path.exists(dst):
        shutil.copy(masked_img_src[fn], dst)
    boxes_by_image[new_fn] = masked_boxes[fn]
    cats_by_image[new_fn] = masked_cats[fn]
    img_meta[new_fn] = masked_meta[fn]
    n_merged += 1

print(f'masked pool {n_merged}장 리네임 병합 완료 (스테이징: {MASKED_STAGE_DIR})')
print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [7-2. 데이터 제외] task2-masked와 동일한 제외 목록을 반드시 그대로 적용합니다.
#  ⚠ fold_split_masked.json은 이 제외가 적용된 파일 집합 기준으로 만들어졌으므로,
#     목록이 다르면 9번 셀의 "분할-데이터 일치" assert에서 중단됩니다.
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png',
]

for fn in EXCLUDE_FILES:
    if fn in boxes_by_image:
        for d in (boxes_by_image, cats_by_image, img_meta):
            d.pop(fn, None)
        print('이미지 제외:', fn)
    else:
        print('⚠ 목록에 없는 파일명(오타 확인):', fn)

print(f"제외 처리 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [8. 74종 라벨 매핑] Task2와 동일한 체계: train 56종 -> 1~56, test 전용 18종 -> 57~74.
#  fold 구성을 Task2와 맞추려면 이 매핑도 완전히 동일해야 합니다 (make_folds의 층화 라벨 계산에 사용됨).
cat2label = {int(c['name']): c['id'] for c in pool_coco['categories'] if c['id'] != 0}
label2cat = {v: k for k, v in cat2label.items()}
all_cats = [label2cat[l] for l in sorted(label2cat)]

for i, c in enumerate(sorted(train_cats), start=1):
    assert cat2label.get(c) == i, f'train 클래스 {c}가 라벨 {cat2label.get(c)}에 매핑됨 (기대: {i})'

merged_cats = {c for cs in cats_by_image.values() for c in cs}
missing = sorted(merged_cats - set(cat2label))
assert not missing, f'매핑에 없는 클래스 존재: {missing}'

test_only_labels = sorted(set(cat2label.values()) - set(range(1, len(train_cats) + 1)))
print(f'전체 {len(cat2label)}종 매핑 | train {len(train_cats)}종 -> 1~{len(train_cats)}'
      f' | test 전용 라벨: {test_only_labels}')


In [ ]:
# [9. 5-fold 분할 로드 + fold별 클래스 배분 점검] - 고정 분할 (재계산 없음)
#  StratifiedGroupKFold를 다시 돌리지 않고, task2-masked에서 export한 fold_split_masked.json을
#  그대로 로드합니다. 이유:
#   1) fold-matched WBF: task2-masked RF-DETR 체크포인트와 fold별 valid 이미지 집합이 동일해야 함
#   2) 분할 재계산은 세션/계정 간 sklearn·numpy 버전 차이로 다른 분할이 나올 위험이 있음
with open(FOLD_SPLIT_JSON, encoding='utf-8') as f:
    fixed_split = json.load(f)

file_names = sorted(boxes_by_image)
name_to_idx = {fn: i for i, fn in enumerate(file_names)}

folds = []
for fi in range(config['data']['n_splits']):
    tr_names = fixed_split[f'fold{fi}']['train']
    va_names = fixed_split[f'fold{fi}']['valid']
    # 안전장치: 저장된 분할이 이번 세션의 병합 데이터와 완전히 일치하는지 확인
    #  (masked/pool2 파일 목록·제외 목록이 export 시점과 다르면 여기서 바로 에러로 잡힘)
    only_in_split = (set(tr_names) | set(va_names)) - set(file_names)
    only_in_data = set(file_names) - (set(tr_names) | set(va_names))
    assert not only_in_split and not only_in_data, (
        f'fold{fi} 분할-데이터 불일치\n'
        f'  분할에만 있는 파일 예: {sorted(only_in_split)[:5]}\n'
        f'  데이터에만 있는 파일 예: {sorted(only_in_data)[:5]}')
    folds.append((np.array([name_to_idx[fn] for fn in tr_names]),
                  np.array([name_to_idx[fn] for fn in va_names])))
print('고정 fold 분할 로드 완료 (재계산 없음):', FOLD_SPLIT_JSON)

# 그룹 누수 점검: 같은 그룹('msk_' 접두어 제거 기준 구성 코드)이 train/valid 양쪽에 있으면 안 됩니다.
_g = lambda fn: (fn[len('msk_'):] if fn.startswith('msk_') else fn).split('_0_2')[0]
for fi, (tr, va) in enumerate(folds):
    leak = {_g(file_names[i]) for i in tr} & {_g(file_names[i]) for i in va}
    assert not leak, f'fold {fi} 그룹 누수: {sorted(leak)[:5]}'
print('그룹 누수 없음 (masked/원본 동일 구성은 항상 같은 fold 쪽)')

def summarize_fold_distribution(folds, file_names, cats_by_image, cat2label):
    """fold별 train/valid 이미지·박스 수와 클래스 커버리지를 점검합니다. (로컬 정의 - 저장소에 없음)"""
    all_labels = set(cat2label.values())
    rows, val_pivot = [], {}
    for fi, (tr, va) in enumerate(folds):
        def label_box_counts(idxs):
            cnt = defaultdict(int)
            for i in idxs:
                for c in cats_by_image[file_names[i]]:
                    cnt[cat2label[c]] += 1
            return cnt
        tr_cnt, va_cnt = label_box_counts(tr), label_box_counts(va)
        rows.append({
            'fold': fi,
            'train_imgs': len(tr), 'valid_imgs': len(va),
            'train_boxes': sum(tr_cnt.values()), 'valid_boxes': sum(va_cnt.values()),
            'train_missing_labels': sorted(all_labels - set(tr_cnt)),
            'valid_missing_labels': sorted(all_labels - set(va_cnt)),
        })
        val_pivot[f'fold{fi}_valid'] = va_cnt
    summary = pd.DataFrame(rows)
    valid_pivot = pd.DataFrame(val_pivot).reindex(sorted(all_labels)).fillna(0).astype(int)
    valid_pivot.index.name = 'label'
    return summary, valid_pivot

fold_summary, valid_pivot = summarize_fold_distribution(folds, file_names, cats_by_image, cat2label)
display(fold_summary)

for _, r in fold_summary.iterrows():
    if r['train_missing_labels']:
        print(f"⚠ fold {r['fold']}: train에 없는 라벨 {r['train_missing_labels']}")
    if r['valid_missing_labels']:
        print(f"(참고) fold {r['fold']}: valid에 없는 라벨 {r['valid_missing_labels']}")

valid_pivot   # 라벨 x fold별 valid 박스 수 (셀 마지막 줄 -> 표로 표시. 12번 셀 클래스별 집계에서 재사용)


In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [10. COCO 포맷 fold 디렉토리 생성] Task2와 동일한 저장소 함수 조합입니다.
#  이 산출물(fold{i}/{train,valid}/_annotations.coco.json + 이미지)을 다음 셀에서 YOLO 포맷으로 변환합니다.
cache_images(TRAIN_IMG_DIR, config['data']['cache_dir'])
cache_images(POOL_IMG_DIR, config['data']['cache_dir'])
cache_images(MASKED_STAGE_DIR, config['data']['cache_dir'])   # masked 리네임 사본(msk_*.png)도 같은 캐시에 추가

write_fold_dirs(folds, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

# fold 디렉토리 복사가 끝나면 캐시는 더 필요 없으므로 삭제해 디스크를 확보합니다
#  (masked pool 추가로 데이터가 커져 fold 5개 사본 + 캐시가 공존하면 디스크가 부족할 수 있음)
shutil.rmtree(config['data']['cache_dir'], ignore_errors=True)
print('이미지 캐시 삭제 완료 (fold 디렉토리에 이미 복사됨)')

save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('COCO 포맷 fold 디렉토리 생성 완료:', config['data']['dataset_dir'])


## 11. YOLO 포맷 변환

Ultralytics는 객체 검출 학습에 COCO json을 직접 쓰지 않고 `images/` + `labels/`(YOLO txt) 구조와
`data.yaml`을 요구한다 (확인 결과: 세그멘테이션/포즈가 아닌 detection 학습에 COCO json을 그대로 읽는
네이티브 경로는 없음). 다만 공식 변환 유틸 `ultralytics.data.converter.convert_coco()`가 이 변환을
대신해주므로 직접 파싱하지 않고 그대로 사용한다.

- `cls91to80=False`로 호출하면 `class_index = category_id - 1`을 그대로 쓴다 (COCO 80/91클래스
  리매핑을 켜지 않음 — 우리 데이터는 원래 COCO 80종이 아니므로 반드시 꺼야 함).
- 저장소 `build_coco()`가 넣는 더미 배경 카테고리(`id=0`, name='pill')에는 박스가 하나도 없으므로,
  별도로 제거하지 않아도 실제 74종이 정확히 YOLO class index `0~73`으로 매핑된다.
- `convert_coco()`는 라벨 txt만 생성하고 이미지는 복사하지 않으므로, 이미지는 심볼릭 링크로 연결해
  디스크 중복을 피한다.
- 변환된 txt 파일명은 COCO json의 `file_name`과 동일한 stem을 쓰므로, `images/`와 `labels/`를
  같은 파일명 규칙으로 나란히 두면 Ultralytics가 자동으로 짝을 찾는다.


In [ ]:
# [11. YOLO 포맷 변환 실행]
def build_yolo_fold(fold_idx, coco_dataset_dir, yolo_dataset_dir):
    """fold{fold_idx}의 COCO 포맷(train/valid)을 YOLO 포맷(images/labels)으로 변환합니다.

    Args:
        fold_idx (int): fold 번호
        coco_dataset_dir (str): write_fold_dirs()의 output_dir (COCO 포맷 fold 루트)
        yolo_dataset_dir (str): YOLO 포맷 fold를 생성할 루트

    Returns:
        str: 이 fold의 data.yaml 경로
    """
    fold_root = os.path.join(yolo_dataset_dir, f'fold{fold_idx}')

    for split in ('train', 'valid'):
        coco_split_dir = os.path.join(coco_dataset_dir, f'fold{fold_idx}', split)
        img_dst = os.path.join(fold_root, split, 'images')
        lbl_dst = os.path.join(fold_root, split, 'labels')
        os.makedirs(img_dst, exist_ok=True)
        os.makedirs(lbl_dst, exist_ok=True)

        # 1) 이미지: 복사 대신 심볼릭 링크 (COCO 포맷 fold 디렉토리와 디스크 중복 방지)
        for src in glob.glob(os.path.join(coco_split_dir, '*.png')):
            link = os.path.join(img_dst, os.path.basename(src))
            if not os.path.exists(link):
                os.symlink(os.path.abspath(src), link)

        # 2) 라벨: convert_coco()로 COCO json -> YOLO txt 변환 후 이동
        #    labels_dir은 *.json을 비재귀 탐색하므로, json 1개만 있는 split 폴더를 그대로 넘기면 됩니다.
        tmp_convert_dir = os.path.join(yolo_dataset_dir, '_convert_tmp', f'fold{fold_idx}_{split}')
        shutil.rmtree(tmp_convert_dir, ignore_errors=True)
        convert_coco(labels_dir=coco_split_dir, save_dir=tmp_convert_dir,
                    use_segments=False, use_keypoints=False, cls91to80=False)

        for txt_path in glob.glob(os.path.join(tmp_convert_dir, 'labels', '*', '*.txt')):
            shutil.move(txt_path, os.path.join(lbl_dst, os.path.basename(txt_path)))
        shutil.rmtree(tmp_convert_dir, ignore_errors=True)

        n_imgs = len(glob.glob(os.path.join(img_dst, '*.png')))
        n_lbls = len(glob.glob(os.path.join(lbl_dst, '*.txt')))
        print(f'fold{fold_idx}/{split}: 이미지 {n_imgs} / 라벨 {n_lbls}')

    # 3) data.yaml: names는 YOLO class index(0-based) -> 원본 category_id 문자열
    #    (class_index = category_id - 1 이므로, label(1~74)과 index+1이 정확히 대응)
    names = {i: str(label2cat[i + 1]) for i in range(len(label2cat))}
    yaml_path = os.path.join(fold_root, 'data.yaml')
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump({
            'path': os.path.abspath(fold_root),
            'train': 'train/images',
            'val': 'valid/images',
            'names': names,
        }, f, allow_unicode=True, sort_keys=False)
    return yaml_path

fold_yaml_paths = [
    build_yolo_fold(fi, config['data']['dataset_dir'], config['data']['yolo_dataset_dir'])
    for fi in range(config['data']['n_splits'])
]
print('data.yaml 경로:')
for p in fold_yaml_paths:
    print(' ', p)


In [ ]:
# [12. YOLOv8 모델 헬퍼] 저장소 model.py의 RF-DETR variant 매핑과 같은 패턴입니다 (라이브러리가 달라 별도 정의).
#  ultralytics 패키지(8.x)는 yolov8*/yolo11* 가중치를 모두 지원하므로 파일명만 v8 계열로 바꾸면 됩니다.
_YOLO_VARIANTS = {
    'nano': 'yolov8n.pt', 'small': 'yolov8s.pt', 'medium': 'yolov8m.pt',
    'large': 'yolov8l.pt', 'xlarge': 'yolov8x.pt',
}

def get_yolo_model(variant='medium', checkpoint_path=None):
    """YOLOv8 모델을 생성합니다. checkpoint_path가 주어지면 그 가중치로 로드합니다."""
    weights = checkpoint_path or _YOLO_VARIANTS.get(variant.lower())
    if weights is None:
        raise ValueError(f"알 수 없는 YOLOv8 variant: {variant} (지원: {list(_YOLO_VARIANTS)})")
    return YOLO(weights)


In [ ]:
# [13. fold 학습 + 리포팅] 저장소 train.py의 train_fold()/report_fold_result() 패턴을 YOLO용으로 재정의합니다.
#  - backup_dir(Drive)에 이미 {tag}_fold{i}_best.pt가 있으면 학습을 건너뜁니다 (이어하기).
#  - loss 가중치는 Ultralytics 내장 train 인자(box/cls/dfl)로 전달합니다. v8DetectionLoss가
#    box_gain*box + cls_gain*cls + dfl_gain*dfl 가중합을 계산하므로 loss 코드 수정이 필요 없습니다.
#  - Ultralytics는 학습 종료 시 results.csv/results.png(학습 곡선)를 자체 생성하므로 그대로 백업합니다.
def train_fold_yolo(fold_idx, fold_yaml, model_variant, model_tag, train_cfg,
                    local_output_dir, backup_dir):
    """fold 하나를 학습하고 best 체크포인트를 backup_dir에 복사합니다. 백업이 이미 있으면 건너뜁니다."""
    exp = f'{model_tag}_fold{fold_idx}'
    dst = os.path.join(backup_dir, f'{exp}_best.pt')

    if os.path.exists(dst):
        print(f'[fold {fold_idx}] 백업 존재 → 건너뜀')
        return dst

    print(f"\n{'='*50}\n[fold {fold_idx}] 학습 시작\n{'='*50}")
    model = get_yolo_model(model_variant)
    model.train(
        data=fold_yaml,
        epochs=train_cfg['epochs'],
        imgsz=train_cfg['imgsz'],
        batch=train_cfg['batch'],
        patience=train_cfg['patience'],
        seed=train_cfg['seed'],
        box=train_cfg['box_gain'],
        cls=train_cfg['cls_gain'],   # 기본 0.5 -> 1.5: 분류 오류 패널티 강화 (WBF 앙상블에서 분류 담당)
        dfl=train_cfg['dfl_gain'],
        project=local_output_dir,
        name=exp,
        exist_ok=True,
        plots=True,
    )

    os.makedirs(backup_dir, exist_ok=True)
    run_dir = os.path.join(local_output_dir, exp)
    src = os.path.join(run_dir, 'weights', 'best.pt')
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'[fold {fold_idx}] best 백업 → {dst}')
    else:
        dst = None
        print(f'[fold {fold_idx}] best.pt 없음 — 백업 실패')

    for fn in ('results.csv', 'results.png'):
        p = os.path.join(run_dir, fn)
        if os.path.exists(p):
            shutil.copy(p, os.path.join(backup_dir, f'{exp}_{fn}'))

    del model
    torch.cuda.empty_cache()
    return dst


def report_fold_result_yolo(fold_idx, checkpoint_path, fold_yaml):
    """fold 하나의 valid mAP를 계산합니다. 학습을 건너뛴 fold도 체크포인트를 다시 로드해 평가합니다.

    Returns:
        dict: {'map'(mAP@0.5:0.95), 'map_50', 'map_per_class'(class index 0~nc-1 정렬 배열), 'names'}
    """
    model = get_yolo_model(checkpoint_path=checkpoint_path)
    metrics = model.val(data=fold_yaml, split='val', plots=False, verbose=False)
    result = {
        'map': float(metrics.box.map),
        'map_50': float(metrics.box.map50),
        'map_per_class': np.asarray(metrics.box.maps, dtype=float),
        'names': metrics.names,
    }
    del model
    torch.cuda.empty_cache()
    return result


## 14. 지정 fold 학습 실행

**fold0는 Colab에서 학습 중**이므로 이 노트북은 `FOLDS_THIS_SESSION`에 지정한 fold만 학습합니다.

- 세션당 fold 1개 권장: `FOLDS_THIS_SESSION = [1]` → 커밋 → 다음 세션 `[2]` → ...
  (yolov8m 100ep 기준 fold 1개가 수 시간이므로, 2개를 넣으면 12h를 넘길 수 있습니다.
  Colab fold0 소요 시간을 보고 세션당 개수를 정하세요)
- `train_fold_yolo()`는 `backup_dir`에 해당 fold의 `*_best.pt`가 이미 있으면 자동 스킵합니다.
  이미 완료한 fold의 커밋 Output을 Input으로 추가하고 아래 선택 셀로 복사해두면,
  실수로 같은 fold를 지정해도 다시 학습하지 않습니다.
- 학습이 끝나면 fold별 valid mAP가 출력되고, `results.csv`/`results.png`(학습곡선)도 함께 백업됩니다.


In [ ]:
# (선택) 이어하기: 이전 커밋 output(outputs/*_best.pt)을 Input으로 추가했다면 backup_dir로 복사
#  -> train_fold_yolo()가 이미 학습된 fold를 자동으로 건너뜁니다.
# PREV_OUTPUT_DIR = '/kaggle/input/<이전-커밋-슬러그>/outputs'
# for p in glob.glob(os.path.join(PREV_OUTPUT_DIR, '*_best.pt')):
#     shutil.copy(p, config['output']['backup_dir'])
#     print('복사:', os.path.basename(p))


In [ ]:
# [14. 지정 fold 학습 + 리포팅 실행]
FOLDS_THIS_SESSION = [1]   # 이번 세션에 학습할 fold 번호 목록 (fold0는 Colab 담당)

assert all(0 <= fi < config['data']['n_splits'] for fi in FOLDS_THIS_SESSION), 'fold 번호 범위 확인'

def run_folds_yolo(config, fold_yaml_paths, fold_indices):
    """지정한 fold들만 학습+리포팅합니다 (Colab판 run_kfold_yolo의 fold 지정 버전)."""
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
    checkpoints, fold_metrics = {}, {}
    for fi in fold_indices:
        dst = train_fold_yolo(
            fold_idx=fi,
            fold_yaml=fold_yaml_paths[fi],
            model_variant=config['model']['variant'],
            model_tag=config['model']['tag'],
            train_cfg=config['train'],
            local_output_dir=config['output']['local_output_dir'],
            backup_dir=config['output']['backup_dir'],
        )
        checkpoints[fi] = dst
        if dst is None:
            print(f'[fold {fi}] 체크포인트 없음 — 리포팅 생략')
            continue
        metrics = report_fold_result_yolo(fi, dst, fold_yaml_paths[fi])
        fold_metrics[fi] = metrics
        print(f"[fold {fi}] 완료 | mAP@0.5:0.95: {metrics['map']:.4f} | mAP@0.5: {metrics['map_50']:.4f}")
    print(f'\n▶ 이번 세션 fold {fold_indices} 완료')
    return {'checkpoints': checkpoints, 'fold_metrics': fold_metrics}

run_result = run_folds_yolo(config, fold_yaml_paths, FOLDS_THIS_SESSION)
print('\nbackup_dir 내용:', sorted(os.listdir(config['output']['backup_dir'])))


## 산출물

- `/kaggle/working/outputs/yolov8m_task4_syn74_masked_fold{i}_best.pt` — 이번 세션에 학습한 fold 체크포인트
  (**Colab판과 동일한 파일명 체계** — Colab의 fold0 + Kaggle의 fold1~4를 한 폴더에 모으면
  fold-matched WBF 앙상블 노트북에서 그대로 사용 가능)
- `..._fold{i}_results.csv` / `_results.png` — fold별 학습 곡선 (Ultralytics 자동 생성)
- 이 노트북은 체크포인트 분담 생산이 목적이라 test 추론/WBF/CSV는 수행하지 않습니다
  (Colab판 또는 별도 앙상블 노트북에서 진행).

**다음 세션**: 이 커밋의 Output을 Input으로 추가 → 선택 셀에서 복사 → `FOLDS_THIS_SESSION`을 다음 fold로 변경 → Save & Run All
